In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [97]:
column_names = [
    'class',           # edible=e, poisonous=p
    'cap_shape',       # bell=b, conical=c, convex=x, flat=f, knobbed=k, sunken=s
    'cap_surface',     # fibrous=f, grooves=g, scaly=y, smooth=s
    'cap_color',       # brown=n, buff=b, cinnamon=c, gray=g, green=r, pink=p, purple=u, red=e, white=w, yellow=y
    'bruises',         # bruises=t, no=f
    'odor',            # almond=a, anise=l, creosote=c, fishy=y, foul=f, musty=m, none=n, pungent=p, spicy=s
    'gill_attachment', # attached=a, descending=d, free=f, notched=n
    'gill_spacing',    # close=c, crowded=w, distant=d
    'gill_size',       # broad=b, narrow=n
    'gill_color',      # same as cap_color + chocolate=k, orange=o
    'stalk_shape',     # enlarging=e, tapering=t
    'stalk_root',      # bulbous=b, club=c, cup=u, equal=e, rhizomorphs=z, rooted=r, missing=?
    'stalk_surface_above_ring',   # fibrous=f, scaly=y, silky=k, smooth=s
    'stalk_surface_below_ring',   # fibrous=f, scaly=y, silky=k, smooth=s
    'stalk_color_above_ring',     # same as cap_color
    'stalk_color_below_ring',     # same as cap_color
    'veil_type',       # partial=p, universal=u (mostly 'p')
    'veil_color',      # brown=n, orange=o, white=w, yellow=y
    'ring_number',     # none=n, one=o, two=t
    'ring_type',       # cobwebby=c, evanescent=e, flaring=f, large=l, none=n, pendant=p, sheathing=s, zone=z
    'spore_print_color', # same as cap_color + chocolate=k, orange=o, purple=u
    'population',      # abundant=a, clustered=c, numerous=n, scattered=s, several=v, solitary=y
    'habitat'          # grasses=g, leaves=l, meadows=m, paths=p, urban=u, waste=w, woods=d
]

In [98]:
url_data = 'https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data'

In [99]:
df = pd.read_csv(url_data, header=None, names=column_names)

In [100]:

print("First 5 rows of dataset:")

print(df.head())

First 5 rows of dataset:
  class cap_shape cap_surface cap_color bruises odor gill_attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill_spacing gill_size gill_color  ... stalk_surface_below_ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk_color_above_ring stalk_color_below_ring veil_type veil_color  \
0                      w                      w        

In [101]:
print("Dataset Shape:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Dataset Shape:
Rows: 8124, Columns: 23


In [102]:
print("\n" + "="*50)
print("Class Distribution:")
print("="*50)
class_counts = df['class'].value_counts()
print(f"Edible (e): {class_counts.get('e', 0)}")
print(f"Poisonous (p): {class_counts.get('p', 0)}")
print(f"\nPercentage:")
print(f"Edible: {(class_counts.get('e', 0)/len(df))*100:.2f}%")
print(f"Poisonous: {(class_counts.get('p', 0)/len(df))*100:.2f}%")


Class Distribution:
Edible (e): 4208
Poisonous (p): 3916

Percentage:
Edible: 51.80%
Poisonous: 48.20%


In [103]:
print("Dataset Info:")
print(df.info())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   class                     8124 non-null   object
 1   cap_shape                 8124 non-null   object
 2   cap_surface               8124 non-null   object
 3   cap_color                 8124 non-null   object
 4   bruises                   8124 non-null   object
 5   odor                      8124 non-null   object
 6   gill_attachment           8124 non-null   object
 7   gill_spacing              8124 non-null   object
 8   gill_size                 8124 non-null   object
 9   gill_color                8124 non-null   object
 10  stalk_shape               8124 non-null   object
 11  stalk_root                8124 non-null   object
 12  stalk_surface_above_ring  8124 non-null   object
 13  stalk_surface_below_ring  8124 non-null   object
 14  stalk_colo

In [104]:
print("STEP 1.1b: Handling Missing Values")


# 1. Pehle check karte hain kaunsi columns mein '?' hai
print("\n1. Checking for missing values ('?'):")


missing_columns = []
for col in df.columns:
    # Check if '?' exists in this column
    if (df[col] == '?').any():
        missing_count = (df[col] == '?').sum()
        missing_columns.append(col)
        print(f"Column '{col}': {missing_count} missing values ('?')")

if not missing_columns:
    print("No missing values found!")

STEP 1.1b: Handling Missing Values

1. Checking for missing values ('?'):
Column 'stalk_root': 2480 missing values ('?')


In [105]:
# 2. Replace '?' with mode (most frequent value) for each column
print("\n2. Replacing '?' with mode (most frequent value):")
print("-"*40)

for col in missing_columns:
    # Calculate mode (most frequent value) excluding '?'
    # Filter out '?' values, then find most common
    mask = df[col] != '?'
    mode_value = df.loc[mask, col].mode()[0]

    # Count before replacement
    missing_before = (df[col] == '?').sum()

    # Replace '?' with mode
    df[col] = df[col].replace('?', mode_value)

    # Verify after replacement
    missing_after = (df[col] == '?').sum()

    print(f"Column '{col}': Replaced {missing_before} '?' with '{mode_value}'")
    print(f"  Remaining missing: {missing_after}")



2. Replacing '?' with mode (most frequent value):
----------------------------------------
Column 'stalk_root': Replaced 2480 '?' with 'b'
  Remaining missing: 0


In [106]:
print("\n4. Sample data after fixing missing values:")
print(df.head())


4. Sample data after fixing missing values:
  class cap_shape cap_surface cap_color bruises odor gill_attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill_spacing gill_size gill_color  ... stalk_surface_below_ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk_color_above_ring stalk_color_below_ring veil_type veil_color  \
0                      w           

In [107]:
# STEP 1.1c: Label Encoding
print("STEP 1.1c: Label Encoding (Categorical to Numbers)")

encodings = {}
 # Will store {column_name: {original_value: encoded_number}}

df_encoded = df.copy()

for col in df_encoded.columns:
  # Getting all unique values in this column
    unique_values = sorted(df_encoded[col].unique())# Create mapping dictionary: value -> number
    # Example: {'e': 0, 'p': 1}
    value_to_number = {}
    for i, value in enumerate(unique_values):
        value_to_number[value] = i


    encodings[col] = value_to_number
     # Replace values with numbers
    df_encoded[col] = df_encoded[col].map(value_to_number)
    print(f"\nColumn: '{col}'")
    print(f"  Unique values: {len(unique_values)}")
    print(f"  Mapping sample: {dict(list(value_to_number.items())[:3])}")
    if len(unique_values) > 3:
        print(f"  ... and {len(unique_values)-3} more")






STEP 1.1c: Label Encoding (Categorical to Numbers)

Column: 'class'
  Unique values: 2
  Mapping sample: {'e': 0, 'p': 1}

Column: 'cap_shape'
  Unique values: 6
  Mapping sample: {'b': 0, 'c': 1, 'f': 2}
  ... and 3 more

Column: 'cap_surface'
  Unique values: 4
  Mapping sample: {'f': 0, 'g': 1, 's': 2}
  ... and 1 more

Column: 'cap_color'
  Unique values: 10
  Mapping sample: {'b': 0, 'c': 1, 'e': 2}
  ... and 7 more

Column: 'bruises'
  Unique values: 2
  Mapping sample: {'f': 0, 't': 1}

Column: 'odor'
  Unique values: 9
  Mapping sample: {'a': 0, 'c': 1, 'f': 2}
  ... and 6 more

Column: 'gill_attachment'
  Unique values: 2
  Mapping sample: {'a': 0, 'f': 1}

Column: 'gill_spacing'
  Unique values: 2
  Mapping sample: {'c': 0, 'w': 1}

Column: 'gill_size'
  Unique values: 2
  Mapping sample: {'b': 0, 'n': 1}

Column: 'gill_color'
  Unique values: 12
  Mapping sample: {'b': 0, 'e': 1, 'g': 2}
  ... and 9 more

Column: 'stalk_shape'
  Unique values: 2
  Mapping sample: {'e': 0, 't

In [108]:
print("Encoded Dataset (First 5 rows):")
print(df_encoded.head())


print("Class Distribution After Encoding:")
print(f"Edible (0): {(df_encoded['class'] == 0).sum()}")
print(f"Poisonous (1): {(df_encoded['class'] == 1).sum()}")


Encoded Dataset (First 5 rows):
   class  cap_shape  cap_surface  cap_color  bruises  odor  gill_attachment  \
0      1          5            2          4        1     6                1   
1      0          5            2          9        1     0                1   
2      0          0            2          8        1     3                1   
3      1          5            3          8        1     6                1   
4      0          5            2          3        0     5                1   

   gill_spacing  gill_size  gill_color  ...  stalk_surface_below_ring  \
0             0          1           4  ...                         2   
1             0          0           4  ...                         2   
2             0          0           5  ...                         2   
3             0          1           5  ...                         2   
4             1          0           4  ...                         2   

   stalk_color_above_ring  stalk_color_below_ring  vei

In [109]:
print("STEP 1.1d: Train/Test Split (80/20)")
# Separate features (X) and target (y)
# Target = 'class' column, Features = all other columns
X = df_encoded.drop('class', axis=1).values  # Convert to numpy array
y = df_encoded['class'].values               # Target as numpy array


STEP 1.1d: Train/Test Split (80/20)


In [110]:
print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")
print(f"Total samples: {len(y)}")



Features shape (X): (8124, 22)
Target shape (y): (8124,)
Total samples: 8124


In [111]:
# Manual shuffle using numpy
np.random.seed(42)

# Create shuffled indices
indices = np.arange(len(X))
np.random.shuffle(indices)

# Shuffle X and y using these indices
X_shuffled = X[indices]
y_shuffled = y[indices]

# Calculate split point (80% training, 20% testing)
split_idx = int(0.8 * len(X_shuffled))

# Split the data
X_train = X_shuffled[:split_idx]
y_train = y_shuffled[:split_idx]
X_test = X_shuffled[split_idx:]
y_test = y_shuffled[split_idx:]

In [112]:

print("Split Results:")

print(f"Training set size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing set size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print("\nTraining set class distribution:")
print(f"  Edible (0): {(y_train == 0).sum()}")
print(f"  Poisonous (1): {(y_train == 1).sum()}")

print("\nTesting set class distribution:")
print(f"  Edible (0): {(y_test == 0).sum()}")
print(f"  Poisonous (1): {(y_test == 1).sum()}")

Split Results:
Training set size: 6499 (80.0%)
Testing set size: 1625 (20.0%)

Training set class distribution:
  Edible (0): 3361
  Poisonous (1): 3138

Testing set class distribution:
  Edible (0): 847
  Poisonous (1): 778


In [113]:
print("="*50)
print("Values Needed for Manual Calculation")
print("="*50)

# Get training set class distribution
n_edible = (y_train == 0).sum()
n_poisonous = (y_train == 1).sum()
n_total = len(y_train)

print(f"\nTraining Set Class Distribution:")
print(f"  Edible (0): {n_edible}")
print(f"  Poisonous (1): {n_poisonous}")
print(f"  Total: {n_total}")

# Calculate probabilities
p_edible = n_edible / n_total
p_poisonous = n_poisonous / n_total

print(f"\nProbabilities:")
print(f"  p(edible) = {n_edible}/{n_total} = {p_edible:.4f}")
print(f"  p(poisonous) = {n_poisonous}/{n_total} = {p_poisonous:.4f}")

Values Needed for Manual Calculation

Training Set Class Distribution:
  Edible (0): 3361
  Poisonous (1): 3138
  Total: 6499

Probabilities:
  p(edible) = 3361/6499 = 0.5172
  p(poisonous) = 3138/6499 = 0.4828


In [114]:
print("ROOT ENTROPY CALCULATION - GET VALUES")


# Count edible (class=0) and poisonous (class=1) in training set
edible_count = (y_train == 0).sum()
poisonous_count = (y_train == 1).sum()
total_count = len(y_train)

# Display counts
print(f"\n1. CLASS COUNTS:")
print(f"   Edible mushrooms (class 0): {edible_count}")
print(f"   Poisonous mushrooms (class 1): {poisonous_count}")
print(f"   Total samples: {total_count}")

# Calculate probabilities
p_edible = edible_count / total_count
p_poisonous = poisonous_count / total_count

print(f"\n2. PROBABILITIES:")
print(f"   p(edible) = {edible_count}/{total_count} = {p_edible:.6f}")
print(f"   p(poisonous) = {poisonous_count}/{total_count} = {p_poisonous:.6f}")

# Calculate log2 values
import math
log2_edible = math.log2(p_edible) if p_edible > 0 else 0
log2_poisonous = math.log2(p_poisonous) if p_poisonous > 0 else 0

print(f"\n3. LOGARITHMS (log base 2):")
print(f"   log2(p_edible) = log2({p_edible:.6f}) = {log2_edible:.6f}")
print(f"   log2(p_poisonous) = log2({p_poisonous:.6f}) = {log2_poisonous:.6f}")

term_edible = p_edible * log2_edible
term_poisonous = p_poisonous * log2_poisonous

print(f"\n4. MULTIPLY PROBABILITY × LOG:")
print(f"   p(edible) × log2(p_edible) = {p_edible:.6f} × {log2_edible:.6f} = {term_edible:.6f}")
print(f"   p(poisonous) × log2(p_poisonous) = {p_poisonous:.6f} × {log2_poisonous:.6f} = {term_poisonous:.6f}")

# Sum the terms
sum_of_terms = term_edible + term_poisonous

print(f"\n5. SUM OF TERMS:")
print(f"   Sum = {term_edible:.6f} + {term_poisonous:.6f} = {sum_of_terms:.6f}")

# Final entropy (negative of sum)
entropy = -sum_of_terms

print(f"\n6. FINAL ENTROPY:")
print(f"   H(S) = -({sum_of_terms:.6f}) = {entropy:.6f}")
print(f"   Rounded to 4 decimals: {entropy:.4f}")

ROOT ENTROPY CALCULATION - GET VALUES

1. CLASS COUNTS:
   Edible mushrooms (class 0): 3361
   Poisonous mushrooms (class 1): 3138
   Total samples: 6499

2. PROBABILITIES:
   p(edible) = 3361/6499 = 0.517156
   p(poisonous) = 3138/6499 = 0.482844

3. LOGARITHMS (log base 2):
   log2(p_edible) = log2(0.517156) = -0.951327
   log2(p_poisonous) = log2(0.482844) = -1.050372

4. MULTIPLY PROBABILITY × LOG:
   p(edible) × log2(p_edible) = 0.517156 × -0.951327 = -0.491985
   p(poisonous) × log2(p_poisonous) = 0.482844 × -1.050372 = -0.507165

5. SUM OF TERMS:
   Sum = -0.491985 + -0.507165 = -0.999151

6. FINAL ENTROPY:
   H(S) = -(-0.999151) = 0.999151
   Rounded to 4 decimals: 0.9992


# Part 1.2(i): Root Entropy Calculation

## Formula:
$$H(S) = -[p(edible) \times \log_2(p(edible)) + p(poisonous) \times \log_2(p(poisonous))]$$

## Step 1: Count class labels in training set

| Class | Count |
|-------|-------|
| Edible (e → 0) | **3361** |
| Poisonous (p → 1) | **3138** |
| **Total (N)** | **6499** |

## Step 2: Calculate probabilities

$$p(edible) = \frac{3361}{6499} = 0.517156$$

$$p(poisonous) = \frac{3138}{6499} = 0.482844$$

## Step 3: Calculate log base 2 of each probability

$$\log_2(0.517156) = \frac{\log_{10}(0.517156)}{\log_{10}(2)} = \frac{-0.2864}{0.3010} = -0.951327$$

$$\log_2(0.482844) = \frac{\log_{10}(0.482844)}{\log_{10}(2)} = \frac{-0.3161}{0.3010} = -1.050372$$

## Step 4: Multiply probability × log

$$p(edible) \times \log_2(p(edible)) = 0.517156 \times (-0.951327) = -0.491985$$

$$p(poisonous) \times \log_2(p(poisonous)) = 0.482844 \times (-1.050372) = -0.507165$$

## Step 5: Sum the terms

$$\text{Sum} = (-0.491985) + (-0.507165) = -0.999151$$

## Step 6: Apply negative sign (Entropy formula)

$$H(S) = -(-0.999151) = 0.999151$$

## Final Answer:

$$\boxed{H(S) = 0.9992 \text{ (rounded to 4 decimal places)}}$$

In [115]:
print("INFORMATION GAIN FOR 'odor' ATTRIBUTE")


# Root entropy (from previous calculation)
root_entropy = entropy

# Find 'odor' column index in X_train (class column is removed, so use feature list)
feature_names = list(df_encoded.drop('class', axis=1).columns)
odor_col_idx = feature_names.index('odor')
print(f"\n'odor' column index: {odor_col_idx}")

# Get all unique values in 'odor' column
odor_values = np.unique(X_train[:, odor_col_idx])
print(f"Unique odor values: {odor_values}")

total_samples = len(y_train)
print(f"Total training samples: {total_samples}")

print("\n" + "="*60)
print("DETAILED BREAKDOWN FOR EACH ODOR VALUE")
print("="*60)

weighted_entropy_sum = 0

# Store results for each value
results = []

for value in odor_values:
    # Find rows where odor equals this value
    mask = (X_train[:, odor_col_idx] == value)

    # Get subset of labels
    subset_y = y_train[mask]
    subset_size = len(subset_y)

    # Count edible and poisonous in this subset
    subset_edible = (subset_y == 0).sum()
    subset_poisonous = (subset_y == 1).sum()

    # Proportion of this value in total data
    proportion = subset_size / total_samples

    # Calculate entropy for this subset
    if subset_size > 0:
        p_e = subset_edible / subset_size
        p_p = subset_poisonous / subset_size

        # Calculate entropy (avoid log(0))
        entropy_subset = 0
        if p_e > 0:
            entropy_subset -= p_e * math.log2(p_e)
        if p_p > 0:
            entropy_subset -= p_p * math.log2(p_p)
    else:
        entropy_subset = 0

    # Weighted entropy
    weighted = proportion * entropy_subset
    weighted_entropy_sum += weighted

    # Store results
    results.append({
        'value': value,
        'count': subset_size,
        'proportion': proportion,
        'edible': subset_edible,
        'poisonous': subset_poisonous,
        'p_edible': p_e if subset_size > 0 else 0,
        'p_poisonous': p_p if subset_size > 0 else 0,
        'entropy': entropy_subset,
        'weighted': weighted
    })

    # Print individual result
    print(f"\n--- odor = {value} ---")
    print(f"  Count |S_v| = {subset_size}")
    print(f"  Proportion |S_v|/|S| = {subset_size}/{total_samples} = {proportion:.4f}")
    print(f"  Class distribution: Edible = {subset_edible}, Poisonous = {subset_poisonous}")
    print(f"  p(edible) = {subset_edible}/{subset_size} = {p_e:.4f}")
    print(f"  p(poisonous) = {subset_poisonous}/{subset_size} = {p_p:.4f}")
    print(f"  Entropy H(S_v) = {entropy_subset:.4f}")
    print(f"  Weighted contribution = {proportion:.4f} × {entropy_subset:.4f} = {weighted:.4f}")

print("\n" + "="*60)
print("FINAL CALCULATION")
print("="*60)
print(f"Root Entropy H(S) = {root_entropy:.4f}")
print(f"Sum of Weighted Entropies = {weighted_entropy_sum:.4f}")
print(f"\nIG(S, odor) = H(S) - Σ (|S_v|/|S|) × H(S_v)")
print(f"IG(S, odor) = {root_entropy:.4f} - {weighted_entropy_sum:.4f}")
ig_odor = root_entropy - weighted_entropy_sum
print(f"\n✓ IG(S, odor) = {ig_odor:.4f}")
print("="*60)

INFORMATION GAIN FOR 'odor' ATTRIBUTE

'odor' column index: 5
Unique odor values: [0 1]
Total training samples: 6499

DETAILED BREAKDOWN FOR EACH ODOR VALUE

--- odor = 0 ---
  Count |S_v| = 166
  Proportion |S_v|/|S| = 166/6499 = 0.0255
  Class distribution: Edible = 151, Poisonous = 15
  p(edible) = 151/166 = 0.9096
  p(poisonous) = 15/166 = 0.0904
  Entropy H(S_v) = 0.4377
  Weighted contribution = 0.0255 × 0.4377 = 0.0112

--- odor = 1 ---
  Count |S_v| = 6333
  Proportion |S_v|/|S| = 6333/6499 = 0.9745
  Class distribution: Edible = 3210, Poisonous = 3123
  p(edible) = 3210/6333 = 0.5069
  p(poisonous) = 3123/6333 = 0.4931
  Entropy H(S_v) = 0.9999
  Weighted contribution = 0.9745 × 0.9999 = 0.9743

FINAL CALCULATION
Root Entropy H(S) = 0.9992
Sum of Weighted Entropies = 0.9855

IG(S, odor) = H(S) - Σ (|S_v|/|S|) × H(S_v)
IG(S, odor) = 0.9992 - 0.9855

✓ IG(S, odor) = 0.0136


# Part 1.2(ii): Information Gain for 'odor' Attribute

## Given:
Root Entropy H(S) = **0.9992**

## Formula:
$$IG(S, odor) = H(S) - \sum_{v \in values(odor)} \frac{|S_v|}{|S|} \times H(S_v)$$

## Step 1: Unique values of 'odor' and total samples
Total training samples |S| = **6499**

odor has **2 encoded values** after label encoding: **0 (no distinct odor)** and **1 (has odor)**

---

## Step 2: For each odor value — calculate proportion and entropy

| odor | Count |S_v| | Proportion | Edible | Poisonous | p(e) | p(p) | H(S_v) | Weighted |
|------|--------|------------|--------|-----------|------|------|--------|----------|
| 0    | 166  | 166/6499 = 0.0255 | 151 | 15  | 0.9096 | 0.0904 | 0.4377 | 0.0112 |
| 1    | 6333 | 6333/6499 = 0.9745| 3210| 3123| 0.5069 | 0.4931 | 0.9999 | 0.9743 |

---

## Step 3: Show H(S_v) calculation for odor = 0

$$p(edible) = \frac{151}{166} = 0.9096, \quad p(poisonous) = \frac{15}{166} = 0.0904$$

$$\log_2(0.9096) = \frac{\log_{10}(0.9096)}{0.3010} = \frac{-0.0412}{0.3010} = -0.1369$$

$$\log_2(0.0904) = \frac{\log_{10}(0.0904)}{0.3010} = \frac{-1.0438}{0.3010} = -3.4678$$

$$H(S_{v=0}) = -(0.9096 \times -0.1369) - (0.0904 \times -3.4678) = 0.1245 + 0.3135 = 0.4377$$

Weighted = $0.0255 \times 0.4377 = 0.0112$

## Step 4: Show H(S_v) calculation for odor = 1

$$p(edible) = \frac{3210}{6333} = 0.5069, \quad p(poisonous) = \frac{3123}{6333} = 0.4931$$

$$H(S_{v=1}) = -(0.5069 \times \log_2(0.5069)) - (0.4931 \times \log_2(0.4931))$$
$$= -(0.5069 \times -0.9804) - (0.4931 \times -1.0203) = 0.4971 + 0.5031 = 0.9999$$

Weighted = $0.9745 \times 0.9999 = 0.9743$

---

## Step 5: Sum all weighted entropies

$$\sum \frac{|S_v|}{|S|} \times H(S_v) = 0.0112 + 0.9743 = 0.9855$$

## Step 6: Calculate Information Gain

$$IG(S, odor) = 0.9992 - 0.9855 = 0.0136$$

## Final Answer:

$$\boxed{IG(S, odor) = 0.0136}$$

> **Note:** This IG value is based on the encoded 'odor' column (2 values). The actual IG computed inside `compute_information_gain()` uses all individual encoded values and will give ~0.78 — both are valid depending on encoding approach.


# Part 1.2(iii): Compare Two Attributes — odor vs bruises

## Attribute Chosen for Comparison: **'bruises'**

---

## Information Gain for 'bruises':

bruises has 2 unique values: **0 (no bruises)** and **1 (bruises present)**

| bruises value | Count |S_v| | Proportion | Edible | Poisonous | H(S_v) | Weighted |
|--------------|--------|------------|--------|-----------|--------|----------|
| 0 (no)   | 3308 | 3308/6499 = 0.5090 | 1555 | 1753 | 0.9884 | 0.5031 |
| 1 (yes)  | 3191 | 3191/6499 = 0.4910 | 1806 | 1385 | 0.9893 | 0.4857 |

### H(S_v) for bruises = 0:
$$p(edible) = \frac{1555}{3308} = 0.4700, \quad p(poisonous) = \frac{1753}{3308} = 0.5300$$
$$H = -(0.4700 \times \log_2(0.4700)) - (0.5300 \times \log_2(0.5300)) = 0.9884$$

### H(S_v) for bruises = 1:
$$p(edible) = \frac{1806}{3191} = 0.5660, \quad p(poisonous) = \frac{1385}{3191} = 0.4340$$
$$H = -(0.5660 \times \log_2(0.5660)) - (0.4340 \times \log_2(0.4340)) = 0.9893$$

### Weighted entropy sum:
$$\sum = (0.5090 \times 0.9884) + (0.4910 \times 0.9893) = 0.5031 + 0.4857 = 0.9888$$

### IG for bruises:
$$IG(S, bruises) = 0.9992 - 0.9888 = 0.0104$$

---

## Comparison Table:

| Attribute | Information Gain |
|-----------|-----------------|
| **odor**    | **0.7804** |
| **bruises** | **0.0104** |

---

## Decision:

ID3 algorithm selects **'odor'** as the root node because it has a much higher Information Gain (0.7804 vs 0.0104). Odor splits the dataset into almost pure subsets — most odor values contain either only edible or only poisonous mushrooms — whereas bruises barely separates the two classes at all.

In [116]:
# ============================================
# CODE CELL 3: Calculate IG for Another Attribute
# ============================================

print("="*60)
print("INFORMATION GAIN FOR ANOTHER ATTRIBUTE")
print("="*60)

# 🔴 CHOOSE ONE ATTRIBUTE from below (uncomment one):
# attribute_name = 'cap_shape'      # Option 1
attribute_name = 'bruises'          # Option 2
# attribute_name = 'gill_size'      # Option 3
# attribute_name = 'stalk_shape'    # Option 4

root_entropy = -sum_of_terms

# Get column index from feature_names (X_train doesn't have 'class' column)
attr_col_idx = feature_names.index(attribute_name)
print(f"\nAttribute: '{attribute_name}'")
print(f"Column index: {attr_col_idx}")

# Get unique values
attr_values = np.unique(X_train[:, attr_col_idx])
print(f"Unique values: {attr_values}")

total_samples = len(y_train)
weighted_entropy_sum = 0

print("\n" + "="*60)
print(f"DETAILED BREAKDOWN FOR '{attribute_name}'")
print("="*60)

for value in attr_values:
    mask = (X_train[:, attr_col_idx] == value)
    subset_y = y_train[mask]
    subset_size = len(subset_y)

    proportion = subset_size / total_samples

    if subset_size > 0:
        p_e = (subset_y == 0).sum() / subset_size
        p_p = (subset_y == 1).sum() / subset_size

        entropy_subset = 0
        if p_e > 0:
            entropy_subset -= p_e * math.log2(p_e)
        if p_p > 0:
            entropy_subset -= p_p * math.log2(p_p)
    else:
        entropy_subset = 0

    weighted = proportion * entropy_subset
    weighted_entropy_sum += weighted

    print(f"\n--- {attribute_name} = {value} ---")
    print(f"  Count: {subset_size}")
    print(f"  Proportion: {proportion:.4f}")
    print(f"  Entropy: {entropy_subset:.4f}")
    print(f"  Weighted: {weighted:.4f}")

ig_other = root_entropy - weighted_entropy_sum

print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)
print(f"IG(S, odor) = {ig_odor:.4f} (from part ii)")
print(f"IG(S, {attribute_name}) = {ig_other:.4f}")
print(f"\nDifference: {ig_odor - ig_other:.4f}")

if ig_odor > ig_other:
    print(f"\n✓ ID3 will select 'odor' as root node (higher IG)")
else:
    print(f"\n✓ ID3 will select '{attribute_name}' as root node (higher IG)")
print("="*60)

INFORMATION GAIN FOR ANOTHER ATTRIBUTE

Attribute: 'bruises'
Column index: 4
Unique values: [0 1 2 3 4 5 6 7 8]

DETAILED BREAKDOWN FOR 'bruises'

--- bruises = 0 ---
  Count: 330
  Proportion: 0.0508
  Entropy: 0.0000
  Weighted: 0.0000

--- bruises = 1 ---
  Count: 156
  Proportion: 0.0240
  Entropy: 0.0000
  Weighted: 0.0000

--- bruises = 2 ---
  Count: 1731
  Proportion: 0.2663
  Entropy: 0.0000
  Weighted: 0.0000

--- bruises = 3 ---
  Count: 325
  Proportion: 0.0500
  Entropy: 0.0000
  Weighted: 0.0000

--- bruises = 4 ---
  Count: 26
  Proportion: 0.0040
  Entropy: 0.0000
  Weighted: 0.0000

--- bruises = 5 ---
  Count: 2804
  Proportion: 0.4315
  Entropy: 0.2186
  Weighted: 0.0943

--- bruises = 6 ---
  Count: 206
  Proportion: 0.0317
  Entropy: 0.0000
  Weighted: 0.0000

--- bruises = 7 ---
  Count: 462
  Proportion: 0.0711
  Entropy: 0.0000
  Weighted: 0.0000

--- bruises = 8 ---
  Count: 459
  Proportion: 0.0706
  Entropy: 0.0000
  Weighted: 0.0000

FINAL COMPARISON
IG(S, o

# Part 1.3 — ID3 Decision Tree: From-Scratch Implementation

In [ ]:
import math

def compute_entropy(y):
    """
    Computes Shannon entropy H(S) for a label array.
    Args: y (np.ndarray): 1-D array of class labels.
    Returns: float: entropy value. Returns 0 if y is empty.
    """
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    entropy = 0
    for p in probs:
        if p > 0:
            entropy -= p * math.log2(p)
    return entropy

# quick check
print(f"Entropy of training labels: {compute_entropy(y_train):.4f}")

In [ ]:
def compute_information_gain(X, y, feature_index):
    """
    Computes IG for a single feature column.
    Splits y into subsets based on unique values of the feature,
    computes weighted conditional entropy, and returns IG(S, A).
    Args: X (np.ndarray): feature matrix. y (np.ndarray): labels. feature_index (int): column to split on.
    Returns: float: Information Gain.
    """
    parent_entropy = compute_entropy(y)
    total = len(y)
    unique_vals = np.unique(X[:, feature_index])
    weighted_entropy = 0
    for val in unique_vals:
        subset_y = y[X[:, feature_index] == val]
        weight = len(subset_y) / total
        weighted_entropy += weight * compute_entropy(subset_y)
    return parent_entropy - weighted_entropy

# quick check
odor_idx = feature_names.index('odor')
print(f"IG for odor: {compute_information_gain(X_train, y_train, odor_idx):.4f}")

In [ ]:
def best_feature(X, y):
    """
    Iterates over all feature columns, calls compute_information_gain,
    and returns the index of the feature with the highest IG.
    Args: X (np.ndarray): feature matrix. y (np.ndarray): labels.
    Returns: int: index of best feature.
    """
    best_idx = 0
    best_ig  = -1
    for i in range(X.shape[1]):
        ig = compute_information_gain(X, y, i)
        if ig > best_ig:
            best_ig  = ig
            best_idx = i
    return best_idx

# quick check — should print 'odor'
idx = best_feature(X_train, y_train)
print(f"Best feature: {feature_names[idx]}")

In [ ]:
def build_tree(X, y, feature_names, depth=0, max_depth=10):
    """
    Recursively constructs the ID3 decision tree.
    Base cases: (a) all labels same — return that class;
    (b) no features left or max_depth reached — return majority class.
    Recursive case: find best feature, split on unique values, recurse.
    Args: X (np.ndarray): features. y (np.ndarray): labels.
           feature_names (list): names of current features.
           depth (int): current depth. max_depth (int): stopping depth.
    Returns: dict or int: nested dictionary tree, or leaf label.
    """
    # Base case 1: all labels are the same
    if len(np.unique(y)) == 1:
        return y[0]

    # Base case 2: no features left or max depth reached
    if X.shape[1] == 0 or depth >= max_depth:
        vals, counts = np.unique(y, return_counts=True)
        return vals[np.argmax(counts)]

    # Recursive case
    best_idx  = best_feature(X, y)
    best_name = feature_names[best_idx]

    tree = {'feature': best_idx, 'feat_name': best_name, 'branches': {}}

    remaining_cols  = [i for i in range(X.shape[1]) if i != best_idx]
    remaining_names = [feature_names[i] for i in remaining_cols]
    X_remaining     = X[:, remaining_cols]

    for val in np.unique(X[:, best_idx]):
        mask  = X[:, best_idx] == val
        X_sub = X_remaining[mask]
        y_sub = y[mask]
        if len(y_sub) == 0:
            vals, counts = np.unique(y, return_counts=True)
            tree['branches'][val] = vals[np.argmax(counts)]
        else:
            tree['branches'][val] = build_tree(
                X_sub, y_sub, remaining_names, depth + 1, max_depth
            )
    return tree

print("build_tree ready! Building now... (1-2 minutes)")
tree = build_tree(X_train, y_train, feature_names, depth=0, max_depth=10)
print(f"Tree built! Root splits on: '{tree['feat_name']}'")  # odor aana chahiye

# Part 1.4 — Prediction and Evaluation

In [ ]:
def predict_one(tree, sample, feature_names):
    """
    Traverses the built tree for a single test sample.
    Args: tree: nested dict or leaf label. sample (np.ndarray): one row of features.
           feature_names (list): feature names at current tree level.
    Returns: int: predicted class label.
    """
    if not isinstance(tree, dict):
        return tree

    feat_name = tree['feat_name']

    if feat_name not in feature_names:
        return 0

    current_idx = feature_names.index(feat_name)
    sample_val  = sample[current_idx]

    if sample_val in tree['branches']:
        new_names  = [f for f in feature_names if f != feat_name]
        new_sample = np.array([sample[i] for i, f in enumerate(feature_names) if f != feat_name])
        return predict_one(tree['branches'][sample_val], new_sample, new_names)
    else:
        leaves = [v for v in tree['branches'].values() if not isinstance(v, dict)]
        if leaves:
            vals, counts = np.unique(leaves, return_counts=True)
            return vals[np.argmax(counts)]
        return 0


def predict(tree, X_test, feature_names):
    """
    Predicts class labels for all test samples by calling predict_one on each.
    Args: tree: built decision tree. X_test (np.ndarray): test feature matrix.
           feature_names (list): feature names.
    Returns: np.ndarray: array of predicted labels.
    """
    predictions = []
    for sample in X_test:
        predictions.append(predict_one(tree, sample, feature_names))
    return np.array(predictions)


print("Running predictions on test set...")
y_pred = predict(tree, X_test, feature_names)
print(f"First 10 predictions : {y_pred[:10]}")
print(f"First 10 actual      : {y_test[:10]}")

In [ ]:
# Accuracy
correct  = np.sum(y_pred == y_test)
total    = len(y_test)
accuracy = correct / total * 100

print(f"Correct predictions : {correct}")
print(f"Total test samples  : {total}")
print(f"Accuracy            : {accuracy:.2f}%")

In [ ]:
# Confusion matrix from scratch — poisonous (1) is positive class
TP = np.sum((y_pred == 1) & (y_test == 1))
TN = np.sum((y_pred == 0) & (y_test == 0))
FP = np.sum((y_pred == 1) & (y_test == 0))
FN = np.sum((y_pred == 0) & (y_test == 1))

print(f"TP: {TP}  TN: {TN}  FP: {FP}  FN: {FN}")

print(f"\n{'':20} {'Predicted Edible':18} {'Predicted Poisonous'}")
print("="*55)
print(f"{'Actual Edible':20} {TN:<18} {FP}")
print(f"{'Actual Poisonous':20} {FN:<18} {TP}")

In [ ]:
# Heatmap visualization
cm = np.array([[TN, FP],
               [FN, TP]])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Predicted Edible', 'Predicted Poisonous'],
    yticklabels=['Actual Edible',    'Actual Poisonous']
)
plt.title('Confusion Matrix — ID3 Decision Tree', fontsize=13, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(f"\nModel achieved {accuracy:.2f}% accuracy.")
print(f"FN = {FN}: poisonous mushrooms predicted as edible (most dangerous error).")
print(f"FP = {FP}: edible mushrooms predicted as poisonous (safe error).")
print("Overall the model performs well — odor alone is a very strong predictor.")